<a href="https://colab.research.google.com/github/leejsya/BreastMNIST/blob/main/BreastMNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Library & Module Import

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2
from torch.utils.data import DataLoader, Subset

#### BreastMNIST 데이터셋 불러오기

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path

# 클래스 0, 1을 각각 레이블로 변경
names = {0: 'malignant', 1: 'benign'}
# .npz 파일 불러오기
data = np.load("breastmnist_224.npz")

for split in ['train', 'val', 'test']:
    for i, (images, labels) in enumerate(zip(data[f"{split}_images"], data[f"{split}_labels"])):
        # ex) breast_cls/train/malignant
        folder = Path("breast_cls/") / split / names[int(labels[0])]
        folder.mkdir(parents=True, exist_ok=True)
        Image.fromarray(images).save(folder / f"{split}_{i:04d}.png")
    print(f"{split}: {len(data[f"{split}_images"])}")


train: 546
val: 78
test: 156


#### 평균, 표준편차 구하기

In [ ]:
# train data의 평균, 표준편차 계산
train_data = data["train_images"].astype(np.float32) / 255.0
mean = train_data.mean()
std = train_data.std()
print(f"mean:{mean:.3f}, std:{std:.3f}")

mean:0.328, std:0.219


#### Transforms

In [ ]:
transforms_train = v2.Compose(
    [
        v2.ToImage(),
        v2.RandomHorizontalFlip(),
        v2.RandomResizedCrop(size=224, scale=(0.8, 1.0)),
        v2.RandomRotation(degrees=10),
        v2.ToDtype(dtype=torch.float32, scale=True),
        v2.Normalize(mean=[mean], std=[std]),
    ]
)

transforms_eval = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float32, scale=True),
        v2.Normalize(mean=[mean], std=[std])
    ]
)


#### Dataset

In [ ]:
train_dataset = ImageFolder(
    root = './breast_cls/train',
    transform = transforms_train,
)

val_dataset = ImageFolder(
    root = './breast_cls/val',
    transform = transforms_eval
)

test_dataset = ImageFolder(
    root = './breast_cls/test',
    transform = transforms_eval,
)

#### DataLoader

In [ ]:
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=42,
    shuffle=True,
)
val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=42,
)
test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=42,
)

#### ImageNet 모델 불러오기

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)

# 백본 전체 고정
for p in model.parameters():
    p.requires_grad = False
# 분류기 클래스 개수를 1로 조정
model.fc = nn.Linear(model.fc.in_features, 1)

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(device)

model.to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer=  optim.Adam(model.parameters())

mps


In [ ]:
print(model.fc)

Linear(in_features=512, out_features=1, bias=True)


#### Evaluation Function

In [ ]:
def evaluate(dataloader, model, loss_fn):
    model.eval()
    losses = []
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in dataloader:
            inputs, labels = batch[0].to(device), batch[1].to(device)
            logits = model(inputs).squeeze(-1)

            loss = loss_fn(logits, labels.float())
            losses.append(loss.item())

            pred = (torch.sigmoid(logits) > 0.5).long()
            correct += (pred == labels).sum().item()
            total += len(logits)

    avg_loss = sum(losses) / len(losses)
    acc = correct / total

    return avg_loss, acc

#### Training Loop Function

In [ ]:
def Train_Loop(dataloader, model, optimizer, epochs, min_loss):
    model.train()
    for epoch in range(epochs):
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d) and not m.weight.requires_grad:
                m.eval()
        step = 0
        for train_batch in dataloader:
            inputs, labels = train_batch[0].to(device), train_batch[1].to(device)
            logits = model(inputs).squeeze(-1)

            loss = loss_fn(logits, labels.float())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            step += 1
            if step % 10 == 0:
                print(f"(step {step}) train loss: {loss.item():.4f}")

        val_loss, val_acc = evaluate(val_dataloader, model, loss_fn)
        print(f"[epoch {epoch+1}/{epochs}] val loss : {val_loss:.4f}, val acc: {val_acc:.4f}\n")

        if min_loss >= val_loss:
            min_loss = val_loss
            torch.save(model.state_dict(), 'model.pt')
    return min_loss

#### FC Layer만 훈련

In [ ]:
min_loss = float('inf')
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
min_loss = Train_Loop(train_dataloader, model, optimizer, epochs=10, min_loss=min_loss)

(step 10) train loss: 0.5505
[epoch 1/10] val loss : 0.5991, val acc: 0.7308

(step 10) train loss: 0.5009
[epoch 2/10] val loss : 0.5734, val acc: 0.7308

(step 10) train loss: 0.6228
[epoch 3/10] val loss : 0.5471, val acc: 0.7308

(step 10) train loss: 0.4572
[epoch 4/10] val loss : 0.5300, val acc: 0.7436

(step 10) train loss: 0.5077
[epoch 5/10] val loss : 0.5238, val acc: 0.7436

(step 10) train loss: 0.4815
[epoch 6/10] val loss : 0.5106, val acc: 0.7436

(step 10) train loss: 0.6031
[epoch 7/10] val loss : 0.4958, val acc: 0.7436

(step 10) train loss: 0.4173
[epoch 8/10] val loss : 0.5257, val acc: 0.7436

(step 10) train loss: 0.5076
[epoch 9/10] val loss : 0.4795, val acc: 0.7436

(step 10) train loss: 0.4138
[epoch 10/10] val loss : 0.4915, val acc: 0.7436



#### Layer4 + FC Layer 훈련

In [ ]:
for p in model.layer4.parameters():
    p.requires_grad = True
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad], lr=1e-4
)
Train_Loop(train_dataloader, model, optimizer, epochs=20, min_loss=min_loss)

(step 10) train loss: 0.2982
[epoch 1/20] val loss : 0.4500, val acc: 0.8333

(step 10) train loss: 0.2306
[epoch 2/20] val loss : 0.3059, val acc: 0.8590

(step 10) train loss: 0.3490
[epoch 3/20] val loss : 0.2305, val acc: 0.9231

(step 10) train loss: 0.2073
[epoch 4/20] val loss : 0.2174, val acc: 0.9103

(step 10) train loss: 0.1043
[epoch 5/20] val loss : 0.4093, val acc: 0.8333

(step 10) train loss: 0.1401
[epoch 6/20] val loss : 0.1802, val acc: 0.9359

(step 10) train loss: 0.1629
[epoch 7/20] val loss : 0.1477, val acc: 0.9487

(step 10) train loss: 0.1197
[epoch 8/20] val loss : 0.1655, val acc: 0.9487

(step 10) train loss: 0.1252
[epoch 9/20] val loss : 0.1726, val acc: 0.9487

(step 10) train loss: 0.1121
[epoch 10/20] val loss : 0.1293, val acc: 0.9744

(step 10) train loss: 0.1527
[epoch 11/20] val loss : 0.1088, val acc: 0.9744

(step 10) train loss: 0.0937
[epoch 12/20] val loss : 0.2227, val acc: 0.8846

(step 10) train loss: 0.3050
[epoch 13/20] val loss : 0.2968,

0.09084483049809933

In [ ]:
model.load_state_dict(torch.load('model.pt', map_location=device))
test_loss, test_acc = evaluate(val_dataloader, model, loss_fn)
print(f"test loss: {test_loss:.4f}, test acc: {test_acc:.4f} ")

test loss: 0.0908, test acc: 0.9872 
